In [ ]:
import pandas as pd

solves = pd.read_csv("results/solves.csv")
cons
print(len(solves)==(8*3*4*20))

In [ ]:
print(8*3*4*20)

In [ ]:
sn_cri_na = solves[solves['solver_name'].str.contains("SNOPT", case=False)]["common_residual_inf"].isna().sum()
ip_cri_na = solves[solves['solver_name'].str.contains("IPOPT", case=False)]["common_residual_inf"].isna().sum()
nl_cri_na = solves[solves['solver_name'].str.contains("NLopt", case=False)]["common_residual_inf"].isna().sum()

print(f"SNOPT common residual NA: {sn_cri_na}") # should be 0
print(f"\nIPOPT common residual NA: {ip_cri_na}") # should be 0
print(f"\nNLopt common residual NA: {nl_cri_na}") # should be >0
print(f"\nnlopt total rows: {len(solves[solves['solver_name'].str.contains("nlopt", case=False)])}")

In [ ]:
snopt_success = solves[(solves['solver_name']=="SNOPT")& (solves['success']==True)]
ipopt_success = solves[(solves['solver_name']=="Ipopt")& (solves['success']==True)]

print(f"snopt residuals sample: \n {snopt_success['common_residual_inf'][0:5]}")

print(f"\nipopt residuals sample: \n {ipopt_success['common_residual_inf'][0:5]}")


In [ ]:
# error rows
error_rows = solves[solves["error_type"].notna()]

print(f"error rows: {len(error_rows)}")
print(error_rows["solver_name"].unique())
print(error_rows["error_type"].unique())
print(error_rows["error_message"].unique())
print(error_rows['orientation_tol'].unique())


In [ ]:
nl = solves[solves.solver_name == "NLopt"]
print(nl.groupby(["orientation_tol", "solution_result"]).size())
print(nl.groupby(["orientation_tol"])[["success", "feasible"]].mean())
print(pd.crosstab(nl.solution_result, nl.error_type.fillna("(clean)")))

In [ ]:
nlopt_nan_ids = set(
    solves[(solves.solver_name == "NLopt") & (solves.error_type == "NonFiniteSolution")].scenario_id
)
others = solves[solves.solver_name.isin(["SNOPT", "Ipopt"]) & solves.scenario_id.isin(nlopt_nan_ids)]
print("SNOPT/IPOPT feasibility on NLopt-NaN scenarios:")
print(others.groupby("solver_name").feasible.mean())

In [ ]:
solves.groupby(["generation_method", "solver_name"]).agg(
    feasible_rate=("feasible", "mean"),
    nan_rate=("error_type", lambda s: (s == "NonFiniteSolution").mean()),
)

In [ ]:
fin = solves[solves.error_type.isna()]                                   # finite returns only
print(fin.groupby("solver_name").feasible.mean())               # feasibility GIVEN finite
print(solves.groupby("solver_name").error_type
        .apply(lambda s: (s == "NonFiniteSolution").mean()))    # NaN rate

print(solves.generation_method.value_counts())

solves.groupby(["generation_method", "solver_name"]).agg(
    feasible_rate=("feasible", "mean"),
    nan_rate=("error_type", lambda s: (s == "NonFiniteSolution").mean()),
)